In [5]:
SEED=15179996

In [1]:
!pip install -q -U transformers peft datasets optuna evaluate scikit-learn accelerate tqdm pandas huggingface_hub torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 49.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is

In [1]:
import torch
import pandas as pd
import numpy as np
import optuna
import evaluate
from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DefaultDataCollator
)
from peft import LoraConfig, get_peft_model
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Dataset

Sentiment140, 5k train, 1k test

https://huggingface.co/datasets/bdanko/sentiment140

In [3]:
# 5k train, 1k for testing
def prepare_dataset(dataset_name="bdanko/sentiment140", train_size=5000, test_size=1000):
    print(f"Loading dataset {dataset_name}...")
    dataset = load_dataset(dataset_name, split="train")
    df = dataset.to_pandas()

    # negative sentiment swapped to 1
    df['sentiment'] = df['sentiment'].replace(4, 1)

    train_df = df.sample(n=train_size, weights=df['sentiment'].map({0: 0.5, 1: 0.5}), random_state=SEED)
    remaining_df = df.drop(train_df.index)

    test_df_neg = remaining_df[remaining_df['sentiment'] == 0].sample(n=test_size // 2, random_state=SEED)
    test_df_pos = remaining_df[remaining_df['sentiment'] == 1].sample(n=test_size // 2, random_state=SEED)
    test_df = pd.concat([test_df_neg, test_df_pos]).sample(frac=1, random_state=SEED)

    return DatasetDict({
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True))
    })

In [6]:
dataset = prepare_dataset()
print(dataset)

Loading dataset bdanko/sentiment140...
DatasetDict({
    train: Dataset({
        features: ['text', 'date', 'user', 'sentiment', 'query'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'date', 'user', 'sentiment', 'query'],
        num_rows: 1000
    })
})


In [7]:
model_id = "mistralai/Mistral-7B-v0.3"
# We stick with the base model,
# because we do not need to care about instruction alignment
# we are training a classifier head that we don't need to explicitly
# care about generation parsing

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [8]:
def tokenize_function(examples):
    texts = [f"Sentiment classification. Text: {text}\nSentiment: " for text in examples["text"]]
    tokenized = tokenizer(texts, truncation=True, max_length=128, padding="max_length")
    tokenized["labels"] = examples["sentiment"]
    return tokenized

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
def get_base_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        dtype=torch.bfloat16,
        device_map={"": 0}, # everything must be on one
                            # GPU and namespace or optuna complains
      )
    model.config.pad_token_id = model.config.eos_token_id
    return model

In [10]:
# huggingface standardized metrics for:
# accuracy
# precision
# recll
# f1

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=predictions, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=predictions, references=labels)["recall"],
        "f1": f1_metric.compute(predictions=predictions, references=labels)["f1"],
    }

In [ ]:
r=32
lr=0.0005
alpha=32

peft_config = LoraConfig(
    r=16, lora_alpha=best_params['alpha'],
    target_modules=["q_proj", "v_proj"], lora_dropout=0.1,
    bias="none", task_type="SEQ_CLS",
)
lr      = best_params['lr']
out_dir = "./best_lora"
hub_id  = "bdanko/mistral-7b-sentiment-lora"

torch.cuda.reset_peak_memory_stats()
model = get_base_model()

model.add_adapter(peft_config, adapter_name="best_adapter")

args = TrainingArguments(
    output_dir=out_dir,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    learning_rate=lr,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False, # causes saving errors
    metric_for_best_model="f1",
    bf16=True,
    report_to="none",
    push_to_hub=True,
    hub_model_id=hub_id,
    #gradient_checkpointing=True,
    #gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

# manual save
trainer.model.save_pretrained(out_dir)

final_metrics = trainer.evaluate()
peak_vram = gpu_memory_gb()
return trainer, final_metrics, peak_vram